In [ ]:
import pandas as pd
import os

def inspect_cataluminescence_data(file_paths):
    for file_path in file_paths:
        if not os.path.exists(file_path):
            print(f"File not found: {os.path.basename(file_path)}\n")
            continue
            
        # Dynamically locate the header row in case these files contain 
        # DAQ970A instrument metadata at the top[cite: 1].
        header_idx = 0
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for idx, line in enumerate(f):
                # Look for common data start markers; adjust if Cataluminescence files use different keys
                if any(key in line for key in ['Scan Num', '101 (', 'Scan Swee', 'Time', 'Timestamp']):
                    header_idx = idx
                    break
                    
        # Load the dataframe skipping the metadata
        df = pd.read_csv(file_path, skiprows=header_idx)
        df.columns = df.columns.str.strip()
        
        # Drop entirely empty rows and columns to get an accurate count
        df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
        
        total_data_points = len(df)
        headers = list(df.columns)
        
        # Identify probable sensor channels (excluding time/index columns)
        time_cols = [col for col in headers if 'time' in col.lower() or 'swee' in col.lower() or 'scan' in col.lower()]
        sensor_channels = [col for col in headers if col not in time_cols]
        
        print(f"FILE: {os.path.basename(file_path)}")
        print(f"Total Data Points (Rows) : {total_data_points}")
        print(f"Total Columns Processed  : {len(headers)}")
        print(f"Suspected Sensor Channels: {len(sensor_channels)}")
        print(f"Headers Found:\n{headers}")
        print("=" * 60 + "\n")

# Execution
target_files = [
    r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\cataluminiscence\20260613T095926_21-11-2025 0 2025-11-21_cleaned.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\cataluminiscence\1781339209_2 2025-11-21 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\cataluminiscence\1781339239_2 2025-11-21 0.csv"
]

inspect_cataluminescence_data(target_files)

In [ ]:
import pandas as pd
import os
import plotly.express as px

file_path = r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\cataluminiscence\20260613T095926_21-11-2025 0 2025-11-21_cleaned.csv"

if os.path.exists(file_path):
    # Load the cleaned CSV directly
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()
    
    # Ensure the sensor channel is treated as numeric data
    df['ch_00'] = pd.to_numeric(df['ch_00'], errors='coerce')
    
    print(f"File successfully loaded: {os.path.basename(file_path)}")
    print(f"Total Rows    : {len(df)}")
    
    # Generate the interactive plot
    fig = px.line(
        df, 
        x='timestamp', 
        y='ch_00', 
        title='Cataluminescence: Raw Sensor Data (ch_00) over Time',
        labels={
            'timestamp': 'Timestamp (mm:ss.s)', 
            'ch_00': 'Temperature (°C)'
        }
    )
    
    # Formatting for a clean, professional look
    fig.update_layout(
        template='plotly_white',
        xaxis_title="Time",
        yaxis_title="Temperature (°C)",
        hovermode="x unified"
    )
    
    fig.show()
else:
    print(f"Error: File not found at {file_path}")

In [ ]:
import pandas as pd
import plotly.express as px
import os

def plot_daq970a_data(file_path):
    # 1. Dynamically locate the start of the data table (bypassing instrument metadata)
    header_idx = 0
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            if 'Scan Swee' in line or 'Scan Number' in line:
                header_idx = idx
                break

    # 2. Load the CSV and drop the empty trailing columns ('Unnamed: X')
    df = pd.read_csv(file_path, skiprows=header_idx)
    df = df.dropna(how='all', axis=1)
    
    # 3. Identify Time and Sensor Columns
    # Grabs explicit timestamp columns like '101Time (Sec)' or 'Scan Sweep Time (Sec)'
    time_cols = [col for col in df.columns if 'time' in col.lower()]
    time_col = time_cols[0] if time_cols else df.columns[0]
    
    # Grabs the temperature channel (e.g., '101 (°C)')
    sensor_cols = [col for col in df.columns if '°c' in col.lower() or '101' in col]
    sensor_col = sensor_cols[-1] if sensor_cols else df.columns[-1]
    
    # 4. Data Conversion
    # Convert temperature to numeric values and timestamps to actual datetime objects
    df[sensor_col] = pd.to_numeric(df[sensor_col], errors='coerce')
    df['Timestamp'] = pd.to_datetime(df[time_col], errors='coerce')
    
    # Clean up any rows that failed conversion
    df_clean = df.dropna(subset=['Timestamp', sensor_col])

    print(f"File loaded successfully: {os.path.basename(file_path)}")
    print(f"Total Valid Data Points: {len(df_clean)}")
    
    # 5. Generate the Interactive Plot
    fig = px.line(
        df_clean,
        x='Timestamp',
        y=sensor_col,
        title=f"Sensor Data over Time: {os.path.basename(file_path)}",
        labels={
            'Timestamp': 'Time (Absolute)',
            sensor_col: 'Temperature (°C)'
        }
    )
    
    fig.update_layout(
        template='plotly_white',
        hovermode='x unified',
        xaxis=dict(rangeslider=dict(visible=True)),
        height=500,
        width=1200
    )
    
    fig.show()

# Execution
file_path = r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\21-11-2025 0 2025-11-21.csv"
if os.path.exists(file_path):
    plot_daq970a_data(file_path)
else:
    print(f"File not found: {file_path}")

In [10]:
import pandas as pd
import plotly.express as px
import os

def plot_daq970a_data(file_path):
    """
    Parses a raw Keysight DAQ970A CSV file, dynamically isolates the time 
    and temperature channels, and generates an interactive line plot.
    """
    # 1. Dynamically locate the start of the data table (bypassing instrument metadata)
    header_idx = 0
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            if 'Scan Swee' in line or 'Scan Number' in line:
                header_idx = idx
                break

    # 2. Load the CSV and drop the empty trailing columns ('Unnamed: X')
    df = pd.read_csv(file_path, skiprows=header_idx)
    df = df.dropna(how='all', axis=1)
    
    # 3. Identify Time and Sensor Columns
    # Grabs explicit timestamp columns like '101Time (Sec)' or 'Scan Sweep Time (Sec)'
    time_cols = [col for col in df.columns if 'time' in col.lower()]
    time_col = time_cols[0] if time_cols else df.columns[0]
    
    # Grabs the temperature channel (e.g., '101 (°C)')
    sensor_cols = [col for col in df.columns if '°c' in col.lower() or '101' in col]
    sensor_col = sensor_cols[-1] if sensor_cols else df.columns[-1]
    
    # 4. Data Conversion
    # Convert temperature to numeric values and timestamps to actual datetime objects
    df[sensor_col] = pd.to_numeric(df[sensor_col], errors='coerce')
    df['Timestamp'] = pd.to_datetime(df[time_col], errors='coerce')
    
    # Clean up any rows that failed conversion
    df_clean = df.dropna(subset=['Timestamp', sensor_col])

    print(f"Plotting: {os.path.basename(file_path)} | Valid Data Points: {len(df_clean)}")
    
    # 5. Generate the Interactive Plot
    fig = px.line(
        df_clean,
        x='Timestamp',
        y=sensor_col,
        title=f"Raw Sensor Data: {os.path.basename(file_path)}",
        labels={
            'Timestamp': 'Time',
            sensor_col: 'Temperature (°C)'
        }
    )
    
    fig.update_layout(
        template='plotly_white',
        hovermode='x unified',
        xaxis=dict(rangeslider=dict(visible=True)),
        height=500,
        width=1200
    )
    
    fig.show()

# =========================================================
# BATCH EXECUTION
# =========================================================
file_paths = [
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\2 2025-11-21 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\03-11-2025 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\03-11-2025 2 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\06-11-2025 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\07-11-2025 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\13-11-2025 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\14-11-2025 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\15-09-2025 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\16-09-2025 0 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\17-11-2025.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\18-09-2025 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\21-11-2025 0 2025-11-21.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\2025-11-15 .csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\2025-11-17 .csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\2025-11-19  0.csv"
]

for fp in file_paths:
    if os.path.exists(fp):
        try:
            plot_daq970a_data(fp)
        except Exception as e:
            print(f"Error processing {os.path.basename(fp)}: {e}")
    else:
        print(f"File not found: {os.path.basename(fp)}")

Plotting: 2 2025-11-21 0.csv | Valid Data Points: 2770


Plotting: 03-11-2025 0.csv | Valid Data Points: 893


Plotting: 03-11-2025 2 0.csv | Valid Data Points: 735


Plotting: 06-11-2025 0.csv | Valid Data Points: 2664


Plotting: 07-11-2025 0.csv | Valid Data Points: 2805


...

Plotting: 18-09-2025 0.csv | Valid Data Points: 910


Plotting: 21-11-2025 0 2025-11-21.csv | Valid Data Points: 10497


Plotting: 2025-11-15 .csv | Valid Data Points: 884


Plotting: 2025-11-17 .csv | Valid Data Points: 424


Plotting: 2025-11-19  0.csv | Valid Data Points: 1255
